# Parashikimi i Vendimeve të Punësimit - JobMatch

## 1. Analiza Eksploruese e të Dhënave (EDA)

# 1. Importimi i librarive

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '6'
# 1. Importimi i librarive te nevojshme
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler

# Klasifikuesit (do te perdoren ne fazat e mevonshme)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# Metrikat e vleresimit
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay)

In [ ]:
# 2. Ngarkimi i te dhenave
df = pd.read_csv("../data/recruitment_data.csv")

In [ ]:
# 3. Shikimi i rreshtave te pare dhe informacioni i dataset-it
print(df.shape)
print(df.head())
print(df.info())

# Kolonat e dataset-it:
# Age                  - Mosha e kandidatit
# Gender               - Gjinia (0 = mashkull, 1 = femer)
# EducationLevel       - Niveli i edukimit (1-4)
# ExperienceYears      - Vitet e përvojes se punes
# PreviousCompanies    - Numri i kompanive te meparshme
# DistanceFromCompany  - Distanca nga kompania (km)
# InterviewScore       - Piket e intervistes (0-100)
# SkillScore           - Piket e aftesive teknike (0-100)
# PersonalityScore     - Piket e personalitetit (0-100)
# RecruitmentStrategy  - Strategjia e rekrutimit (1-3)
# HiringDecision       - Vendimi i punesimit (1 = punesohet, 0 = jo) [TARGET]

In [ ]:
# 4. Verifikimi i cilesise se te dhenave (missing values dhe duplikate)
print("Vlera te munguara per kolone:")
print(df.isnull().sum())
print(f"\nNumri i rreshtave duplikate: {df.duplicated().sum()}")

In [ ]:
# 5. Analiza statistikore pershkruese
df.describe()

In [ ]:
# 6. Shperndarja e variables target (HiringDecision)
print(df['HiringDecision'].value_counts())
print(df['HiringDecision'].value_counts(normalize=True).round(3))

plt.figure(figsize=(6, 4))
sns.countplot(x='HiringDecision', data=df)
plt.title('Shperndarja e vendimeve te punesimit')
plt.xlabel('Vendimi (0 = Jo, 1 = Po)')
plt.ylabel('Numri i kandidateve')
plt.show()

In [ ]:
# 7. Matrica e korelacionit
plt.figure(figsize=(12, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Matrica e Korelacionit')
plt.show()

In [ ]:
# 8. Shperndarja e veçorive numerike sipas vendimit te punesimit
features_num = ['Age', 'ExperienceYears', 'InterviewScore', 'SkillScore', 'PersonalityScore']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flatten(), features_num):
    sns.histplot(data=df, x=col, hue='HiringDecision', kde=True, ax=ax)
    ax.set_title(f'Shperndarja e {col}')
axes.flatten()[-1].axis('off')  
plt.tight_layout()
plt.show()

## 2. Parapërpunimi i të Dhënave

In [ ]:
# Permbledhje e gjetjeve nga EDA
# 
# 1. Dataseti ka 1500 rreshta dhe 11 kolona, te gjitha numerike.
#    Nuk ka vlera te munguara dhe as rreshta duplikate.
#
# 2. Variabla target (HiringDecision) eshte e pabalancuar: 69% e kandidateve
#    nuk punesohen (klasa 0) dhe 31% punesohen (klasa 1). Per kete arsye:
#    - Ndarja train/test do te behet me stratifikim (stratify=y)
#    - Krahas accuracy do te raportojme precision, recall dhe F1-score,
#      sepse accuracy e vetme mund të jete mashtruese me klasa te pabalancuara.
#
# 3. Vecorite kane shkalle shume te ndryshme (p.sh. Age 20-50, scores 0-100,
#    EducationLevel 1-4). Kjo e ben standardizimin (StandardScaler) të
#    domosdoshem per algoritmet e bazuara në distance (KNN) dhe rrjetat neurale.
#
# 4. Nga matrica e korelacionit, RecruitmentStrategy ka lidhjen me te forte
#    me HiringDecision (-0.48), e ndjekur nga EducationLevel (0.24),
#    SkillScore (0.20), PersonalityScore (0.17), InterviewScore (0.15)
#    dhe ExperienceYears (0.12).
#    Age, Gender, PreviousCompanies dhe DistanceFromCompany kane korelacion
#    pothuajse zero - kandidate kryesore per tu hequr ne fazen e feature
#    selection.

In [10]:
# 10. Ndarja e te dhenave ne X (veçorite) dhe y (target)
X = df.drop(columns=['HiringDecision'])
y = df['HiringDecision']

# One-Hot Encoding për RecruitmentStrategy
# Kjo variabel eshte kategorike nominale (1=Agresive, 2=Mesatare, 3=Konservative)
# Vlerat numerike nuk kane renditje kuptimplote, prandaj e kthejme ne kolona binare
# EducationLevel e leme ordinal, sepse ka renditje natyrale (1 < 2 < 3 < 4)
X = pd.get_dummies(X, columns=['RecruitmentStrategy'], prefix='Strategy', drop_first=False)
X = X.astype(float)

print(X.columns.tolist())
print(X.shape)

['Age', 'Gender', 'EducationLevel', 'ExperienceYears', 'PreviousCompanies', 'DistanceFromCompany', 'InterviewScore', 'SkillScore', 'PersonalityScore', 'Strategy_1', 'Strategy_2', 'Strategy_3']
(1500, 12)


In [11]:
# 11. Ndarja ne bashkesi trajnimi dhe testimi (80/20, me stratifikim)
# stratify=y siguron qe raporti 69/31 i klasave te ruhet ne te dyja bashkesite
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Trajnim: {X_train.shape[0]} rreshta")
print(f"Testim:  {X_test.shape[0]} rreshta")
print(f"\nRaporti i klasave ne trajnim:\n{y_train.value_counts(normalize=True).round(3)}")
print(f"\nRaporti i klasave ne testim:\n{y_test.value_counts(normalize=True).round(3)}")

Trajnim: 1200 rreshta
Testim:  300 rreshta

Raporti i klasave ne trajnim:
HiringDecision
0    0.69
1    0.31
Name: proportion, dtype: float64

Raporti i klasave ne testim:
HiringDecision
0    0.69
1    0.31
Name: proportion, dtype: float64


In [12]:
# 12. Standardizimi i veçorive me StandardScaler
# E RENDESISHME: scaler-i ben fit VETEM mbi te dhenat e trajnimit,
# pastaj transformon te dyja bashkesite. Kjo shmang "data leakage" -
# rrjedhjen e informacionit nga testimi ne trajnim.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Versionet e pa-skaluara (X_train, X_test) i ruajme gjithashtu,
# sepse algoritmet e bazuara ne peme nuk kane nevoje per scaling
# dhe do t'i përdorim per nje krahasim me/pa scaling te KNN